In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

: 

In [6]:
crash_df = pd.read_csv(r"C:\Users\Perry\Desktop\Prathamesh\Crash.csv", header=1)
base_df = pd.read_csv(r"C:\Users\Perry\Desktop\Prathamesh\Base.csv", header=4)

crash_df.head(5)

,Type,Aircraft,Aircraft ID,Nation,Commander,Take off Base,Target,Date of Mission,Take off Time,Reason for Crash,Latitude,Longitude,Location
0,Crash,P-51 Mustang,F-6D 44-14,USA,lt.charles Garner,Biak Island,Wama Airfield,4/12/1944,3:00 PM,Bad weather and \nlow fuel,1.7261,128.3408,20 miles south of Morotai Island
1,Crash,P-51 Mustang,"P-51D ""Madam Wham-Dam"" 44-72607",USA,Lt.Col Harvey J Scandrett,North Field (APO 86) (Iwo Jima),Osaka,01-06-1945,7:20 AM,Bad Weather,31,137.0000,Approx 340 km south of Osaka (last radio contact)
2,Crash,P-51 Mustang,P-51D 44-73369,USA,Lt. Paul Ewalt,North Field (APO 86) (Iwo Jima),Kumagaya,03-08-1945,NaN,Anti Aircraft Fire,35.3,139.3000,Crashed Near Oiso
3,Crash,B-17E,B-17E 41-2435,USA,1st Lt William H. Watson,7 Mile Drome (Port Moresby),Australia,02-08-1942,NaN,Japanese Zero,-8.0489,148.1339,"Shot by Japanese zeros, crashed near Mitre roc..."
4,Crash,Curtiss SBC2,SB2C-3 19422,USA,Commander Mark Eslick Jr,USS Intrepid (CV-11),"(Kiirun Harbor, Keelung Harbor)\nNorth Taiwan",12-10-1944,6:16 AM,Unknown,25.1558,121.7544,Crashed near Keelung Harbour Tiwan


In [7]:
# Renaiming columns for consistency
crash_df.columns = ['Type', 'Aircraft', 'Aircraft_ID','Nation', 'Commander','Takeoff_Base','Target','Date_of_Mission','Takeoff_Time','Reason','Latitude','Longitude','Location']
base_df.columns = crash_df.columns

In [8]:
# Dropping columns which are not particulary helpful as prediction features

drop_cols = ['Commander','Location', 'Aircraft_ID']
crash_df = crash_df.drop(columns=drop_cols)
base_df = base_df.drop(columns=drop_cols)

In [9]:
# Converting date column to year 

crash_df['Year'] = pd.to_datetime(crash_df['Date_of_Mission'], format='%d-%m-%Y', errors='coerce').dt.year
base_df['Year']  = pd.to_datetime(base_df['Date_of_Mission'], format='%d-%m-%Y', errors='coerce').dt.year


crash_df.head()

,Type,Aircraft,Nation,Takeoff_Base,Target,Date_of_Mission,Takeoff_Time,Reason,Latitude,Longitude,Year
0,Crash,P-51 Mustang,USA,Biak Island,Wama Airfield,4/12/1944,3:00 PM,Bad weather and \nlow fuel,1.7261,128.3408,NaN
1,Crash,P-51 Mustang,USA,North Field (APO 86) (Iwo Jima),Osaka,01-06-1945,7:20 AM,Bad Weather,31,137.0000,1945.0
2,Crash,P-51 Mustang,USA,North Field (APO 86) (Iwo Jima),Kumagaya,03-08-1945,NaN,Anti Aircraft Fire,35.3,139.3000,1945.0
3,Crash,B-17E,USA,7 Mile Drome (Port Moresby),Australia,02-08-1942,NaN,Japanese Zero,-8.0489,148.1339,1942.0
4,Crash,Curtiss SBC2,USA,USS Intrepid (CV-11),"(Kiirun Harbor, Keelung Harbor)\nNorth Taiwan",12-10-1944,6:16 AM,Unknown,25.1558,121.7544,1944.0


In [10]:
# Labelling crash and safe points 

crash_df['Label'] = 1
base_df['Label'] = 0

df = pd.concat([crash_df, base_df], ignore_index= True)
df

,Type,Aircraft,Nation,Takeoff_Base,Target,Date_of_Mission,Takeoff_Time,Reason,Latitude,Longitude,Year,Label
0,Crash,P-51 Mustang,USA,Biak Island,Wama Airfield,4/12/1944,3:00 PM,Bad weather and \nlow fuel,1.7261,128.3408,NaN,1
1,Crash,P-51 Mustang,USA,North Field (APO 86) (Iwo Jima),Osaka,01-06-1945,7:20 AM,Bad Weather,31,137.0000,1945.0,1
2,Crash,P-51 Mustang,USA,North Field (APO 86) (Iwo Jima),Kumagaya,03-08-1945,NaN,Anti Aircraft Fire,35.3,139.3000,1945.0,1
3,Crash,B-17E,USA,7 Mile Drome (Port Moresby),Australia,02-08-1942,NaN,Japanese Zero,-8.0489,148.1339,1942.0,1
4,Crash,Curtiss SBC2,USA,USS Intrepid (CV-11),"(Kiirun Harbor, Keelung Harbor)\nNorth Taiwan",12-10-1944,6:16 AM,Unknown,25.1558,121.7544,1944.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
93,Base,NIL,Axis (Japan),NIL,NIL,NIL,NIL,NIL,35.2839,139.6677,NaN,0
94,Base,NIL,Axis/Allied (Japanese-held then \nAllied captu...,NIL,NIL,NIL,NIL,NIL,8.72,167.7300,NaN,0
95,Base,NIL,Allied (USA),NIL,NIL,NIL,NIL,NIL,-9.64,160.1500,NaN,0
96,Base,NIL,Allied (USA/Australia),NIL,NIL,NIL,NIL,NIL,-6.33,155.0200,NaN,0


In [11]:
print(df.columns.tolist())
df.columns = df.columns.str.strip()

['Type', 'Aircraft', 'Nation', 'Takeoff_Base', 'Target', 'Date_of_Mission', 'Takeoff_Time', 'Reason', 'Latitude', 'Longitude', 'Year', 'Label']


In [12]:
# Converting catagorical data to numbers

df_final = pd.get_dummies(df, columns=['Aircraft', 'Nation', 'Takeoff_Base', 'Target', 'Target','Reason','Date_of_Mission', 'Takeoff_Time', 'Latitude', 'Longitude'])
df_final.head(5)

,Type,Year,Label,Aircraft_\r\n Lockheed Hudson Mark IVa,Aircraft_ FG-1A Corsair,Aircraft_ B-24 Liberator,Aircraft_ B-24D-125-CO Liberator,Aircraft_ B-25H-5 Mitchell,Aircraft_ Bristol Beaufighter Mark IXc,Aircraft_ C-47A-25-DK Dakota,...,Longitude_155.02,Longitude_156.943333,Longitude_157.076111,Longitude_160.0548,Longitude_160.15,Longitude_166.631012,Longitude_167.166672,Longitude_167.73,Longitude_171.0523,Longitude_171.749722
0,Crash,NaN,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Crash,1945.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Crash,1945.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,Crash,1942.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Crash,1944.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
# Splitting data as train and test 

X = df_final.drop(columns=['Type', 'Label'])
y = df_final['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42)

In [14]:
rf = RandomForestClassifier(n_estimators=100, random_state= 42)
rf.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [15]:
y_pred = rf.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[ 4  0]
 [ 0 16]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00        16

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [16]:
# Predicting crash zones by using data of air bases

crash_small = crash_df[['Latitude', 'Longitude']].copy()
crash_small['Label'] = 1

base_small = base_df[['Latitude', 'Longitude']].copy()
base_small['Label'] = 0

df_small = pd.concat([crash_small, base_small], ignore_index= True)


In [17]:
df_small["Latitude"] = pd.to_numeric(df_small["Latitude"], errors="coerce")
df_small["Longitude"] = pd.to_numeric(df_small["Longitude"], errors="coerce")


In [18]:
X = df_small[["Latitude", "Longitude"]]
y = df_small["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [19]:
base_small["Prediction"] = rf.predict(base_small[["Latitude", "Longitude"]])
base_small["Crash_Probability"] = rf.predict_proba(base_small[["Latitude", "Longitude"]])[:,1]


In [20]:
import folium

m = folium.Map(location=[0, 150], zoom_start=4)

for i, row in base_small.iterrows():
    color = "red" if row["Prediction"] == 1 else "green"
    popup_text = f"Crash Probability: {row['Crash_Probability']:.2f}"
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        popup=popup_text
    ).add_to(m)

m.save("base_predictions_map.html")

m

In [21]:
import numpy as np
import pandas as pd

# Define grid boundaries
lat_range = np.arange(-20, 21, 2)   # -20 to +20, step 2
lon_range = np.arange(120, 171, 2)  # 120 to 170, step 2

# Create grid points
grid_points = []
for lat in lat_range:
    for lon in lon_range:
        grid_points.append({"Latitude": lat, "Longitude": lon})

grid_df = pd.DataFrame(grid_points)


In [22]:
# Predict probabilities for grid points
grid_df["Crash_Probability"] = rf.predict_proba(grid_df[["Latitude", "Longitude"]])[:,1]


In [24]:
import folium
from folium.plugins import HeatMap

# Base map
m = folium.Map(location=[0, 150], zoom_start=4)

# Prepare heatmap data: [lat, lon, weight]
heat_data = [
    [row["Latitude"], row["Longitude"], row["Crash_Probability"]]
    for i, row in grid_df.iterrows()
]

# Add heatmap
HeatMap(heat_data, radius=15, blur=20 ,max_zoom=6).add_to(m)

# Save map
m.save("crash_heatmap.html")

m

In [ ]:
# fix any weird dash characters first
df["Latitude"] = df["Latitude"].astype(str).str.replace("−","-")
df["Longitude"] = df["Longitude"].astype(str).str.replace("−","-")

# convert to float, coerce errors to NaN
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# drop rows where lat or lon is missing
df = df.dropna(subset=["Latitude","Longitude"])

df["Crash_Probability"] = rf.predict_proba(df[["Latitude","Longitude"]])[:,1]

import plotly.express as px

fig = px.density_map(
    df,
    lat='Latitude', lon='Longitude', z='Crash_Probability',
    radius=15,
    center=dict(lat=0, lon=150), zoom=3,
    animation_frame="Aircraft",
    map_style="carto-darkmatter",
    color_continuous_scale="Turbo",
    width=1600,
    height=800
)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()




: 